In [1]:
import sys
from pathlib import Path

# /app is the Docker WORKDIR; when running locally, use the repo root instead.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils" / "spark_session.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env when running locally in Cursor (Docker injects env vars via docker-compose env_file).
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

import os

import yaml
import pyspark.sql.functions as F

from utils.spark_session import create_spark_session

with open(PROJECT_ROOT / "schema.yaml") as f:
    schema = yaml.safe_load(f)

BRONZE_PATH = schema["bronze"]["path"]
LEGAL_BRONZE = f"{BRONZE_PATH}/{schema['bronze']['tables']['legal_docs_raw']['path']}"
WIKI_BRONZE = f"{BRONZE_PATH}/{schema['bronze']['tables']['wiki_docs_raw']['path']}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Legal docs path: {LEGAL_BRONZE}")
print(f"Wiki  docs path: {WIKI_BRONZE}")

# Give Spark more heap for large text tokenization in Docker.
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--driver-memory 4g --executor-memory 4g pyspark-shell",
)

spark = create_spark_session("explore-bronze")
print(f"Spark version: {spark.version}")

Project root: /app
Legal docs path: s3a://cs611-project/bronze/legal_docs_raw
Wiki  docs path: s3a://cs611-project/bronze/wiki_docs_raw


Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fde45941-42a1-4015-ad81-082a3b97b589;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 190ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 

Spark version: 3.5.4


## R2 folder structure

Browse the medallion layers on R2 using Spark's S3A filesystem (same credentials as the pipeline). Run the setup cell above first.

In [2]:
BUCKET_ROOT = f"s3a://{schema['r2']['bucket']}"


def list_r2_tree(spark, path: str, max_depth: int = 2, max_entries: int = 30) -> None:
    """Print folders/files under an s3a:// path (R2)."""
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem

    fs = FileSystem.get(URI(path), sc._jsc.hadoopConfiguration())

    def _walk(current: str, depth: int) -> None:
        p = Path(current)
        if not fs.exists(p):
            print(f"{'  ' * depth}(not found) {current}")
            return

        statuses = list(fs.listStatus(p))
        statuses.sort(key=lambda s: (not s.isDirectory(), s.getPath().getName().lower()))

        shown = 0
        for status in statuses:
            if shown >= max_entries:
                remaining = len(statuses) - shown
                print(f"{'  ' * (depth + 1)}... and {remaining} more")
                break

            name = status.getPath().getName()
            full = status.getPath().toString()
            prefix = "  " * depth

            if status.isDirectory():
                print(f"{prefix}{name}/")
                if depth < max_depth:
                    _walk(full, depth + 1)
            else:
                size_mb = status.getLen() / (1024 * 1024)
                print(f"{prefix}{name}  ({size_mb:.2f} MB)")

            shown += 1

    print(path)
    _walk(path, 0)


# Medallion roots from schema.yaml
LAYER_PATHS = {
    "landing": schema["landing"]["path"],
    "bronze": schema["bronze"]["path"],
    "silver": schema["silver"]["path"],
    "gold": schema["gold"]["path"],
}

print(f"Bucket root: {BUCKET_ROOT}\n")
for layer, path in LAYER_PATHS.items():
    print("=" * 70)
    print(layer)
    print("=" * 70)
    list_r2_tree(spark, path, max_depth=2, max_entries=25)
    print()

# Drill into a specific table (change path / depth as needed)
# list_r2_tree(spark, f"{schema['gold']['path']}/{schema['gold']['tables']['ngram_count']['path']}", max_depth=2)

Bucket root: s3a://cs611-project

landing
s3a://cs611-project/landing
legal_docs/
  drift_data/
    EurLex_snapshot_20050101.csv  (19.79 MB)
    EurLex_snapshot_20060101.csv  (28.20 MB)
    EurLex_snapshot_20070101.csv  (24.92 MB)
    EurLex_snapshot_20080101.csv  (34.70 MB)
    EurLex_snapshot_20090101.csv  (30.85 MB)
    EurLex_snapshot_20100101.csv  (22.78 MB)
    EurLex_snapshot_20110101.csv  (29.10 MB)
    EurLex_snapshot_20120101.csv  (25.94 MB)
    EurLex_snapshot_20130101.csv  (34.31 MB)
    EurLex_snapshot_20140101.csv  (47.16 MB)
    EurLex_snapshot_20150101.csv  (35.35 MB)
    EurLex_snapshot_20160101.csv  (40.48 MB)
    EurLex_snapshot_20170101.csv  (37.47 MB)
    EurLex_snapshot_20180101.csv  (32.65 MB)
    EurLex_snapshot_20190101.csv  (24.23 MB)
  EurLex_snapshot_19870101.csv  (6.44 MB)
  EurLex_snapshot_19880101.csv  (8.06 MB)
  EurLex_snapshot_19890101.csv  (7.17 MB)
  EurLex_snapshot_19900101.csv  (8.83 MB)
  EurLex_snapshot_19910101.csv  (9.64 MB)
  EurLex_snapshot_1

In [3]:
GOLD = schema["gold"]["path"]
MB = schema["model_bank"]["path"]

# Old vs new names — check which exist
candidates = [
    "ngrams", "ngram_count",
    "pos_tags", "pos_counts",
    "pos_tags_wiki", "pos_counts_wiki",
    "labels", "label_store",
    "tfidf_features_train", "dcw_features_train",
    "runs",
]
for name in candidates:
  p = f"{GOLD}/{name}"
  exists = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
      spark._jvm.java.net.URI(p), spark.sparkContext._jsc.hadoopConfiguration()
  ).exists(spark._jvm.org.apache.hadoop.fs.Path(p))
  print(f"{'✓' if exists else '✗'} {p}")

print("\nmodel_bank:")
list_r2_tree(spark, MB, max_depth=3, max_entries=40)

✓ s3a://cs611-project/gold/ngrams
✓ s3a://cs611-project/gold/ngram_count
✓ s3a://cs611-project/gold/pos_tags
✓ s3a://cs611-project/gold/pos_counts
✓ s3a://cs611-project/gold/pos_tags_wiki
✓ s3a://cs611-project/gold/pos_counts_wiki
✗ s3a://cs611-project/gold/labels
✓ s3a://cs611-project/gold/label_store
✓ s3a://cs611-project/gold/tfidf_features_train
✓ s3a://cs611-project/gold/dcw_features_train
✓ s3a://cs611-project/gold/runs

model_bank:
s3a://cs611-project/model_bank
features_extractor/
  dcw_score/
    _delta_log/
      _commits/
      00000000000000000000.json  (0.01 MB)
    part-00000-81b86f57-afea-41f2-b865-3b9ba2adfcab-c000.snappy.parquet  (0.00 MB)
    part-00001-13842b19-2daa-4a00-9d9a-3882e3cce8d1-c000.snappy.parquet  (0.00 MB)
    part-00002-0e4d2ab9-f194-4832-aa88-8dabbbf6a0ac-c000.snappy.parquet  (0.00 MB)
    part-00003-ffe35854-53a2-484a-a538-06c8948a443c-c000.snappy.parquet  (0.00 MB)
    part-00004-bfd501ee-ba8e-4e03-9278-534c5b6d95ed-c000.snappy.parquet  (0.00 MB)
   

In [4]:
list_r2_tree(spark, "s3a://cs611-project/model_bank", max_depth=4)


s3a://cs611-project/model_bank
features_extractor/
  dcw_score/
    _delta_log/
      _commits/
      00000000000000000000.json  (0.01 MB)
    part-00000-81b86f57-afea-41f2-b865-3b9ba2adfcab-c000.snappy.parquet  (0.00 MB)
    part-00001-13842b19-2daa-4a00-9d9a-3882e3cce8d1-c000.snappy.parquet  (0.00 MB)
    part-00002-0e4d2ab9-f194-4832-aa88-8dabbbf6a0ac-c000.snappy.parquet  (0.00 MB)
    part-00003-ffe35854-53a2-484a-a538-06c8948a443c-c000.snappy.parquet  (0.00 MB)
    part-00004-bfd501ee-ba8e-4e03-9278-534c5b6d95ed-c000.snappy.parquet  (0.00 MB)
    part-00005-2d1ffc17-52a1-4448-a90a-621381a2288d-c000.snappy.parquet  (0.00 MB)
    part-00006-d4908293-a957-46bf-ac71-7bdab70bd116-c000.snappy.parquet  (0.00 MB)
    part-00007-68bf77c7-a310-42a4-854a-9360652ed607-c000.snappy.parquet  (0.00 MB)
  dcw_train_doc_ids/
    _delta_log/
      _commits/
      00000000000000000000.json  (0.00 MB)
    part-00000-c7e9838b-0c72-4dd1-977a-b0ed3a75ccd2-c000.snappy.parquet  (0.00 MB)
    part-00001-46c

In [4]:
silver = spark.read.format("delta").load("s3a://cs611-project/silver/legal_docs_processed")
labels = spark.read.format("delta").load("s3a://cs611-project/gold/label_store")
ngrams = spark.read.format("delta").load("s3a://cs611-project/gold/ngrams")

silver_labeled = silver.filter(
    F.col("labels").isNotNull() & (F.length(F.trim("labels")) > 0)
    & F.col("act_raw_text").isNotNull() & (F.length(F.trim("act_raw_text")) > 100)
).select(F.col("CELEX").alias("document_id")).dropDuplicates(["document_id"])

print("Silver (labeled + text>100):", silver_labeled.count())
print("Label store rows:", labels.select("document_id").dropDuplicates().count())
print("Ngrams rows:", ngrams.count())
print("Ngrams / silver eligible:", round(ngrams.count() / silver_labeled.count(), 3))

Silver (labeled + text>100): 28333


Label store rows: 28333
Ngrams rows: 28333


Ngrams / silver eligible: 1.0


### SANITY CHECK FOR GOLD PROCESSING

In [5]:
import pyspark.sql.functions as F

GOLD = schema["gold"]["path"]
SILVER = schema["silver"]["path"]

PATHS = {
    "silver_legal": f"{SILVER}/{schema['silver']['tables']['legal_docs_processed']['path']}",
    "silver_wiki": f"{SILVER}/{schema['silver']['tables']['wiki_docs_processed']['path']}",
    "label_store": f"{GOLD}/{schema['gold']['corpus']['label_store']['path']}",
    "ngrams": f"{GOLD}/{schema['gold']['corpus']['ngrams']['path']}",
    "pos_tags": f"{GOLD}/{schema['gold']['corpus']['pos_tags']['path']}",
    "pos_tags_wiki": f"{GOLD}/{schema['gold']['corpus']['pos_tags_wiki']['path']}",
}

def _load(path: str, name: str):
    try:
        return spark.read.format("delta").load(path)
    except Exception as e:
        print(f"FAIL  Could not read {name}: {path}\n      {e}")
        return None

silver_legal = _load(PATHS["silver_legal"], "silver_legal")
silver_wiki = _load(PATHS["silver_wiki"], "silver_wiki")
labels = _load(PATHS["label_store"], "label_store")
ngrams = _load(PATHS["ngrams"], "ngrams")
pos_legal = _load(PATHS["pos_tags"], "pos_tags")
pos_wiki = _load(PATHS["pos_tags_wiki"], "pos_tags_wiki")

# --- Baseline: what ngram job targets (label + text filters + dedupe) ---
silver_ngram_eligible = (
    silver_legal.filter(
        F.col("labels").isNotNull() & (F.length(F.trim("labels")) > 0)
        & F.col("act_raw_text").isNotNull() & (F.length(F.trim("act_raw_text")) > 100)
    )
    .select(F.col("CELEX").alias("document_id"))
    .dropDuplicates(["document_id"])
)

# POS legal: text > 100 only (no label filter in pos_counts.py)
silver_pos_eligible = (
    silver_legal.filter(
        F.col("act_raw_text").isNotNull() & (F.length(F.trim("act_raw_text")) > 100)
    )
    .select(F.col("CELEX").alias("document_id"))
    .dropDuplicates(["document_id"])
)

silver_wiki_eligible = (
    silver_wiki.filter(
        F.col("text").isNotNull() & (F.length(F.trim("text")) > 100)
    )
    .select(F.col("id").alias("document_id"))
    .dropDuplicates(["document_id"])
)

def _dup_count(df, col="document_id"):
    return (
        df.groupBy(col).count().filter(F.col("count") > 1).count()
        if df is not None else None
    )

def _pct(num, denom):
    return round(100.0 * num / denom, 1) if denom else 0.0

print("=" * 70)
print("ROW COUNTS")
print("=" * 70)

checks = []

if ngrams is not None:
    n_ngrams = ngrams.count()
    n_eligible = silver_ngram_eligible.count()
    ratio = n_ngrams / n_eligible if n_eligible else 0
    print(f"Silver ngram-eligible (label + text>100): {n_eligible:,}")
    print(f"gold/ngrams rows:                         {n_ngrams:,}")
    print(f"Coverage (ngrams / eligible):             {_pct(n_ngrams, n_eligible)}%")
    checks.append(("ngrams coverage >= 95%", ratio >= 0.95))
    checks.append(("ngrams no duplicate document_id", _dup_count(ngrams) == 0))
    checks.append(("ngrams token_count > 0", ngrams.filter(F.col("token_count") <= 0).count() == 0))

if pos_legal is not None:
    n_pos = pos_legal.count()
    n_pos_eligible = silver_pos_eligible.count()
    print(f"\nSilver pos-eligible (text>100):           {n_pos_eligible:,}")
    print(f"gold/pos_tags rows:                       {n_pos:,}")
    print(f"Coverage (pos / eligible):                {_pct(n_pos, n_pos_eligible)}%")
    checks.append(("pos_tags coverage >= 90%", n_pos / n_pos_eligible >= 0.90 if n_pos_eligible else False))
    checks.append(("pos_tags no duplicate document_id", _dup_count(pos_legal) == 0))
    empty_pos = pos_legal.filter(F.col("n_total_tokens") <= 0).count()
    checks.append(("pos_tags n_total_tokens > 0", empty_pos == 0))

if pos_wiki is not None:
    n_wiki_pos = pos_wiki.count()
    n_wiki_eligible = silver_wiki_eligible.count()
    print(f"\nSilver wiki-eligible (text>100):          {n_wiki_eligible:,}")
    print(f"gold/pos_tags_wiki rows:                  {n_wiki_pos:,}")
    print(f"Coverage (wiki pos / eligible):           {_pct(n_wiki_pos, n_wiki_eligible)}%")
    checks.append(("pos_tags_wiki coverage >= 90%", n_wiki_pos / n_wiki_eligible >= 0.90 if n_wiki_eligible else False))
    checks.append(("pos_tags_wiki no duplicate document_id", _dup_count(pos_wiki) == 0))

print("\n" + "=" * 70)
print("CROSS-TABLE (legal)")
print("=" * 70)

if ngrams is not None and pos_legal is not None:
    ngram_ids = ngrams.select("document_id").distinct()
    pos_ids = pos_legal.select("document_id").distinct()

    in_ngrams_not_pos = ngram_ids.join(pos_ids, "document_id", "left_anti").count()
    in_pos_not_ngrams = pos_ids.join(ngram_ids, "document_id", "left_anti").count()
    overlap = ngram_ids.join(pos_ids, "document_id", "inner").count()

    print(f"document_id overlap (ngrams ∩ pos_tags):  {overlap:,}")
    print(f"In ngrams but not pos_tags:               {in_ngrams_not_pos:,}")
    print(f"In pos_tags but not ngrams:                {in_pos_not_ngrams:,}")
    # ngrams is stricter (labels); pos may have more docs
    checks.append(("most ngram docs also in pos_tags", in_ngrams_not_pos / max(ngrams.count(), 1) < 0.05))

if labels is not None and ngrams is not None:
    label_ids = labels.select("document_id").distinct()
    ngrams_not_in_labels = (
        ngrams.select("document_id").distinct()
        .join(label_ids, "document_id", "left_anti")
        .count()
    )
    print(f"ngrams docs missing from label_store:     {ngrams_not_in_labels:,}")
    checks.append(("all ngram docs in label_store", ngrams_not_in_labels == 0))

print("\n" + "=" * 70)
print("QUALITY STATS")
print("=" * 70)

if ngrams is not None:
    print("\n--- ngrams ---")
    ngrams.select(
        F.count("*").alias("rows"),
        F.avg("token_count").alias("avg_token_count"),
        F.expr("percentile(token_count, 0.5)").alias("p50_token_count"),
        F.min("token_count").alias("min_token_count"),
        F.max("token_count").alias("max_token_count"),
    ).show(truncate=False)

    print("Partitions (snapshot_date):")
    ngrams.groupBy("snapshot_date").count().orderBy("snapshot_date").show(5, truncate=False)
    print(f"... total partitions: {ngrams.select('snapshot_date').distinct().count()}")

if pos_legal is not None:
    print("\n--- pos_tags (legal) ---")
    pos_legal.select(
        F.count("*").alias("rows"),
        F.avg("n_total_tokens").alias("avg_n_total_tokens"),
        F.expr("percentile(n_total_tokens, 0.5)").alias("p50_n_total_tokens"),
        F.avg("n_unique_tokens").alias("avg_n_unique_tokens"),
    ).show(truncate=False)

    print("Partitions (snapshot_date):")
    pos_legal.groupBy("snapshot_date").count().orderBy("snapshot_date").show(5, truncate=False)
    print(f"... total partitions: {pos_legal.select('snapshot_date').distinct().count()}")

if pos_wiki is not None:
    print("\n--- pos_tags_wiki ---")
    pos_wiki.select(
        F.count("*").alias("rows"),
        F.avg("n_total_tokens").alias("avg_n_total_tokens"),
        F.expr("percentile(n_total_tokens, 0.5)").alias("p50_n_total_tokens"),
    ).show(truncate=False)

print("\n" + "=" * 70)
print("SCHEMA SAMPLE")
print("=" * 70)
for name, df in [("ngrams", ngrams), ("pos_tags", pos_legal), ("pos_tags_wiki", pos_wiki)]:
    if df is not None:
        print(f"\n{name}:")
        df.printSchema()
        df.select(df.columns[:6]).show(3, truncate=80)

print("\n" + "=" * 70)
print("PASS / FAIL")
print("=" * 70)
for label, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

ROW COUNTS


Silver ngram-eligible (label + text>100): 28,333
gold/ngrams rows:                         28,333
Coverage (ngrams / eligible):             100.0%



Silver pos-eligible (text>100):           28,333
gold/pos_tags rows:                       28,333
Coverage (pos / eligible):                100.0%



Silver wiki-eligible (text>100):          29,963
gold/pos_tags_wiki rows:                  29,963
Coverage (wiki pos / eligible):           100.0%



CROSS-TABLE (legal)


...

... total partitions: 18

--- pos_tags_wiki ---


+-----+------------------+------------------+
|rows |avg_n_total_tokens|p50_n_total_tokens|
+-----+------------------+------------------+
|29963|343.49240730233953|168.0             |
+-----+------------------+------------------+


SCHEMA SAMPLE

ngrams:
root
 |-- document_id: string (nullable = true)
 |-- labels: string (nullable = true)
 |-- snapshot_date: string (nullable = true)
 |-- tokens: string (nullable = true)
 |-- token_count: integer (nullable = true)
 |-- ngram_counts: map (nullable = true)
 |    |-- key: string
 |    |-- value: integer (valueContainsNull = true)
 |-- text_source: string (nullable = true)
 |-- silver_ingest_ts: string (nullable = true)
 |-- silver_source: string (nullable = true)



+--------------+--------------------------------------------------------------------------------+-------------+--------------------------------------------------------------------------------+-----------+--------------------------------------------------------------------------------+
|   document_id|                                                                          labels|snapshot_date|                                                                          tokens|token_count|                                                                    ngram_counts|
+--------------+--------------------------------------------------------------------------------+-------------+--------------------------------------------------------------------------------+-----------+--------------------------------------------------------------------------------+
|    21999D0171|   Transport & Infrastructure; Industry, Technology & IP; International & EU Law|   1999-01-01|avis juridique important 21999d

+-----------+--------------------------------------------------------------------------------+-------------+--------------------------------------------------------------------------------+---------------+--------------+
|document_id|                                                                          labels|snapshot_date|                                                                      pos_counts|n_unique_tokens|n_total_tokens|
+-----------+--------------------------------------------------------------------------------+-------------+--------------------------------------------------------------------------------+---------------+--------------+
| 32002R1061|              Finance, Tax & Banking; Agriculture & Food Safety; Trade & Customs|   2002-01-01|{PRON -> {its -> 2, i -> 11, their -> 1, it -> 1}, AUX -> {be -> 21, may -> 2...|            291|           802|
| 32002D0632|Industry, Technology & IP; International & EU Law; Economic & Competition Policy|   2002-01-01|{PRON ->

+-----------+--------------------------------------------------------------------------------+---------------+--------------+
|document_id|                                                                      pos_counts|n_unique_tokens|n_total_tokens|
+-----------+--------------------------------------------------------------------------------+---------------+--------------+
|   65306067|{DET -> {the -> 12, a -> 2}, ADV -> {also -> 1}, CCONJ -> {and -> 2}, PRON ->...|             78|           172|
|   64527990|{DET -> {the -> 5, this -> 1}, ADV -> {commonly -> 1, highly -> 1}, CCONJ -> ...|             54|            75|
|   65158056|{DET -> {the -> 3, a -> 2}, ADV -> {there -> 1, north -> 1}, CCONJ -> {and ->...|             41|            59|
+-----------+--------------------------------------------------------------------------------+---------------+--------------+
only showing top 3 rows


PASS / FAIL
PASS  ngrams coverage >= 95%
PASS  ngrams no duplicate document_id
PASS  ngrams 

In [2]:
import sys
from pathlib import Path

import pyspark.sql.functions as F

# Match your docker --run-id
RUN = "run001"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils" / "spark_session.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "include" / "gold") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "include" / "gold"))

from run_paths import resolve_feature_run_paths  # noqa: E402

paths = resolve_feature_run_paths(RUN)
GOLD = schema["gold"]["path"]
MB = schema["model_bank"]["path"]

print("Run ID:", RUN)
for k in (
    "ngrams", "labels", "pos_tags",
    "tfidf_train", "tfidf_val_test_oot",
    "dcw_train", "dcw_val_test_oot",
    "tfidf_pkl", "dcw_pkl", "tfidf_json_dir",
    "dcw_score_path", "dcw_train_doc_ids_path",
):
    print(f"  {k}: {paths[k]}")


def path_exists(path: str) -> bool:
    sc = spark.sparkContext
    jvm = sc._jvm
    fs = jvm.org.apache.hadoop.fs.FileSystem.get(
        jvm.java.net.URI(path), sc._jsc.hadoopConfiguration()
    )
    return fs.exists(jvm.org.apache.hadoop.fs.Path(path))


def _load(path: str, name: str):
    try:
        return spark.read.format("delta").load(path)
    except Exception as e:
        print(f"FAIL  Could not read {name}: {path}\n      {e}")
        return None


def _dup_count(df, col="document_id"):
    return df.groupBy(col).count().filter(F.col("count") > 1).count()


# --- Upstream + outputs ---
labels = _load(paths["labels"], "labels")
ngrams = _load(paths["ngrams"], "ngrams")
pos_legal = _load(paths["pos_tags"], "pos_tags")
tfidf_train = _load(paths["tfidf_train"], "tfidf_train")
tfidf_holdout = _load(paths["tfidf_val_test_oot"], "tfidf_val_test_oot")
dcw_train = _load(paths["dcw_train"], "dcw_train")
dcw_holdout = _load(paths["dcw_val_test_oot"], "dcw_val_test_oot")

checks: list[tuple[str, bool]] = []

# --- Expected split sizes from label_store ---
if labels is not None:
    split = labels.select("document_id", "category").dropDuplicates(["document_id"])
    expected_train = split.filter(F.col("category") == "train").count()
    expected_holdout = split.filter(F.col("category") != "train").count()
    print("\n" + "=" * 70)
    print("LABEL SPLITS (label_store)")
    print("=" * 70)
    split.groupBy("category").count().orderBy("category").show()
    print(f"Expected train docs:   {expected_train:,}")
    print(f"Expected holdout docs: {expected_holdout:,}")

print("\n" + "=" * 70)
print("ROW COUNTS")
print("=" * 70)

if tfidf_train is not None and tfidf_holdout is not None:
    n_tr = tfidf_train.count()
    n_ho = tfidf_holdout.count()
    print(f"tfidf_train rows:         {n_tr:,}")
    print(f"tfidf_val_test_oot rows:  {n_ho:,}")
    checks.append(("tfidf_train has rows", n_tr > 0))
    checks.append(("tfidf holdout has rows", n_ho > 0))
    checks.append(("tfidf train ~= label train (±5%)",
                   abs(n_tr - expected_train) / max(expected_train, 1) <= 0.05))
    checks.append(("tfidf holdout ~= label holdout (±5%)",
                   abs(n_ho - expected_holdout) / max(expected_holdout, 1) <= 0.05))
    checks.append(("tfidf_train no duplicate document_id", _dup_count(tfidf_train) == 0))
    checks.append(("tfidf holdout no duplicate document_id", _dup_count(tfidf_holdout) == 0))
    checks.append(("tfidf_train has tfidf + log_tfidf",
                   "tfidf" in tfidf_train.columns and "log_tfidf" in tfidf_train.columns))
    null_tfidf = tfidf_train.filter(F.col("tfidf").isNull()).count()
    checks.append(("tfidf_train no null tfidf vectors", null_tfidf == 0))

if dcw_train is not None and dcw_holdout is not None:
    n_dtr = dcw_train.count()
    n_dho = dcw_holdout.count()
    print(f"dcw_train rows:           {n_dtr:,}")
    print(f"dcw_val_test_oot rows:    {n_dho:,}")
    checks.append(("dcw_train has rows", n_dtr > 0))
    checks.append(("dcw holdout has rows", n_dho > 0))
    checks.append(("dcw train ~= label train (±5%)",
                   abs(n_dtr - expected_train) / max(expected_train, 1) <= 0.05))
    checks.append(("dcw holdout ~= label holdout (±5%)",
                   abs(n_dho - expected_holdout) / max(expected_holdout, 1) <= 0.05))
    checks.append(("dcw_train no duplicate document_id", _dup_count(dcw_train) == 0))
    checks.append(("dcw_train has dcw_features", "dcw_features" in dcw_train.columns))
    empty_dcw = dcw_train.filter(F.size(F.map_keys(F.col("dcw_features"))) == 0).count()
    checks.append(("dcw_train non-empty dcw_features maps", empty_dcw == 0))

print("\n" + "=" * 70)
print("CROSS-TABLE OVERLAP")
print("=" * 70)

if ngrams is not None and tfidf_train is not None:
    ngram_ids = ngrams.select("document_id").distinct()
    tfidf_ids = tfidf_train.select("document_id").union(
        tfidf_holdout.select("document_id")
    ).distinct()
    missing_from_ngrams = tfidf_ids.join(ngram_ids, "document_id", "left_anti").count()
    print(f"tfidf docs not in ngrams: {missing_from_ngrams:,}")
    checks.append(("all tfidf docs come from ngrams", missing_from_ngrams == 0))

if pos_legal is not None and dcw_train is not None:
    pos_ids = pos_legal.select("document_id").distinct()
    dcw_ids = dcw_train.select("document_id").union(
        dcw_holdout.select("document_id")
    ).distinct()
    missing_from_pos = dcw_ids.join(pos_ids, "document_id", "left_anti").count()
    print(f"dcw docs not in pos_tags: {missing_from_pos:,}")
    checks.append(("all dcw docs come from pos_tags", missing_from_pos == 0))

if tfidf_train is not None and dcw_train is not None:
    tfidf_all = tfidf_train.select("document_id").union(
        tfidf_holdout.select("document_id")
    ).distinct()
    dcw_all = dcw_train.select("document_id").union(
        dcw_holdout.select("document_id")
    ).distinct()
    in_tfidf_not_dcw = tfidf_all.join(dcw_all, "document_id", "left_anti").count()
    in_dcw_not_tfidf = dcw_all.join(tfidf_all, "document_id", "left_anti").count()
    print(f"in tfidf but not dcw:     {in_tfidf_not_dcw:,}")
    print(f"in dcw but not tfidf:     {in_dcw_not_tfidf:,}")
    # POS join can differ slightly from ngrams join; small gap OK
    checks.append(("tfidf/dcw doc sets mostly aligned",
                   in_tfidf_not_dcw / max(tfidf_all.count(), 1) < 0.05))

print("\n" + "=" * 70)
print("MODEL BANK ARTIFACTS")
print("=" * 70)

artifact_checks = [
    ("tfidf.pkl", paths["tfidf_pkl"]),
    ("dcw.pkl", paths["dcw_pkl"]),
    ("tfidf JSON dir", paths["tfidf_json_dir"]),
    ("dcw_score Delta", paths["dcw_score_path"]),
    ("dcw_train_doc_ids Delta", paths["dcw_train_doc_ids_path"]),
]
for name, p in artifact_checks:
    ok = path_exists(p)
    print(f"{'OK' if ok else 'MISSING'}  {name}: {p}")
    checks.append((f"{name} exists on R2", ok))

if path_exists(paths["tfidf_json_dir"]):
    meta_path = f"{paths['tfidf_json_dir']}/gold_meta_1_3.json"
    checks.append(("gold_meta_1_3.json exists", path_exists(meta_path)))

if path_exists(paths["dcw_score_path"]):
    dcw_score = _load(paths["dcw_score_path"], "dcw_score")
    if dcw_score is not None:
        n_terms = dcw_score.count()
        print(f"dcw_score terms: {n_terms:,}")
        checks.append(("dcw_score has terms", n_terms > 0))
        dcw_score.select(
            F.count("*").alias("terms"),
            F.avg("score").alias("avg_score"),
            F.min("score").alias("min_score"),
            F.max("score").alias("max_score"),
        ).show()

print("\n" + "=" * 70)
print("SAMPLES")
print("=" * 70)

if tfidf_train is not None:
    print("\n--- tfidf_train ---")
    tfidf_train.printSchema()
    tfidf_train.select("document_id", "tfidf", "log_tfidf").show(3, truncate=60)

if dcw_train is not None:
    print("\n--- dcw_train ---")
    dcw_train.printSchema()
    dcw_train.select("document_id", "dcw_features").show(3, truncate=60)

print("\n" + "=" * 70)
print("PASS / FAIL")
print("=" * 70)
passed = sum(1 for _, ok in checks if ok)
for label, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {label}")
print(f"\n{passed}/{len(checks)} checks passed")

Run ID: run001
  ngrams: s3a://cs611-project/gold/ngrams
  labels: s3a://cs611-project/gold/label_store
  pos_tags: s3a://cs611-project/gold/pos_tags
  tfidf_train: s3a://cs611-project/gold/runs/run001/tfidf_train
  tfidf_val_test_oot: s3a://cs611-project/gold/runs/run001/tfidf_val_test_oot
  dcw_train: s3a://cs611-project/gold/runs/run001/dcw_train
  dcw_val_test_oot: s3a://cs611-project/gold/runs/run001/dcw_val_test_oot
  tfidf_pkl: s3a://cs611-project/model_bank/runs/run001/feature_extractors/tfidf.pkl
  dcw_pkl: s3a://cs611-project/model_bank/runs/run001/feature_extractors/dcw.pkl
  tfidf_json_dir: s3a://cs611-project/model_bank/runs/run001/feature_extractors/tfidf
  dcw_score_path: s3a://cs611-project/model_bank/runs/run001/feature_extractors/dcw_score
  dcw_train_doc_ids_path: s3a://cs611-project/model_bank/runs/run001/feature_extractors/dcw_train_doc_ids



LABEL SPLITS (label_store)


+--------+-----+
|category|count|
+--------+-----+
|     oot| 1687|
|    test| 2668|
|   train|18640|
|     val| 5338|
+--------+-----+

Expected train docs:   18,640
Expected holdout docs: 9,693

ROW COUNTS


tfidf_train rows:         18,640
tfidf_val_test_oot rows:  9,693


dcw_train rows:           18,640
dcw_val_test_oot rows:    9,693


...

in tfidf but not dcw:     0
in dcw but not tfidf:     0



MODEL BANK ARTIFACTS
OK  tfidf.pkl: s3a://cs611-project/model_bank/runs/run001/feature_extractors/tfidf.pkl
OK  dcw.pkl: s3a://cs611-project/model_bank/runs/run001/feature_extractors/dcw.pkl
OK  tfidf JSON dir: s3a://cs611-project/model_bank/runs/run001/feature_extractors/tfidf
OK  dcw_score Delta: s3a://cs611-project/model_bank/runs/run001/feature_extractors/dcw_score
OK  dcw_train_doc_ids Delta: s3a://cs611-project/model_bank/runs/run001/feature_extractors/dcw_train_doc_ids
dcw_score terms: 8,002


+-----+--------------------+--------------------+------------------+
|terms|           avg_score|           min_score|         max_score|
+-----+--------------------+--------------------+------------------+
| 8002|0.002237625443349...|7.009451411100289E-6|0.7639264332351862|
+-----+--------------------+--------------------+------------------+


SAMPLES

--- tfidf_train ---
root
 |-- document_id: string (nullable = true)
 |-- tfidf: vector (nullable = true)
 |-- log_tfidf: vector (nullable = true)
 |-- silver_ingest_ts: string (nullable = true)
 |-- silver_source: string (nullable = true)
 |-- snapshot_date: string (nullable = true)



+-----------+------------------------------------------------------------+------------------------------------------------------------+
|document_id|                                                       tfidf|                                                   log_tfidf|
+-----------+------------------------------------------------------------+------------------------------------------------------------+
| 31989D0092|(50000,[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,2...|(50000,[1,4,5,6,8,10,15,18,31,33,36,37,47,49,52,53,62,68,...|
| 31989D0136|(50000,[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,2...|(50000,[1,4,5,6,8,10,15,24,31,36,37,45,51,58,99,112,117,1...|
| 31989D0224|(50000,[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,2...|(50000,[1,4,5,6,8,10,15,31,36,45,49,67,90,112,114,119,185...|
+-----------+------------------------------------------------------------+------------------------------------------------------------+
only showing top 3 rows


--- dcw_train ---
root

+-----------+------------------------------------------------------------+
|document_id|                                                dcw_features|
+-----------+------------------------------------------------------------+
| 31991R1082|{date -> 0.03814017797510905, remoteness -> 1.40754199492...|
| 31991R3254|{year -> 0.04578862186399953, competition -> 0.0058784027...|
| 31991R2345|{date -> 0.03814017797510905, official -> 0.2490076443389...|
+-----------+------------------------------------------------------------+
only showing top 3 rows


PASS / FAIL
PASS  tfidf_train has rows
PASS  tfidf holdout has rows
PASS  tfidf train ~= label train (±5%)
PASS  tfidf holdout ~= label holdout (±5%)
PASS  tfidf_train no duplicate document_id
PASS  tfidf holdout no duplicate document_id
PASS  tfidf_train has tfidf + log_tfidf
PASS  tfidf_train no null tfidf vectors
PASS  dcw_train has rows
PASS  dcw holdout has rows
PASS  dcw train ~= label train (±5%)
PASS  dcw holdout ~= label holdout (±5%)
P

In [4]:
from pyspark.sql import functions as F

paths = {
    "label_store": "s3a://cs611-project/gold/label_store",
    "embeddings": "s3a://cs611-project/gold/embeddings",
    "tfidf_train": "s3a://cs611-project/gold/runs/run001/tfidf_train",
    "dcw_train": "s3a://cs611-project/gold/runs/run001/dcw_train",
}

for name, path in paths.items():
    df = spark.read.format("delta").load(path)
    print(f"\n{name}: {sorted(df.columns)}")

labels = spark.read.format("delta").load(paths["label_store"])
train = labels.filter(F.col("category") == "train")
print("\nTrain split count:", train.select("document_id").distinct().count())
print("category values:", [r.category for r in labels.select("category").distinct().collect()])


label_store: ['category', 'document_id', 'labels', 'snapshot_date']

embeddings: ['document_id', 'embedding', 'embedding_model', 'labels', 'n_chunks', 'snapshot_date']

tfidf_train: ['document_id', 'log_tfidf', 'silver_ingest_ts', 'silver_source', 'snapshot_date', 'tfidf']

dcw_train: ['dcw_features', 'document_id', 'n_nouns_propn', 'n_nouns_propn_filtered', 'snapshot_date']



Train split count: 18640


category values: ['train', 'test', 'val', 'oot']


In [6]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

LABELS_PATH = f"{schema['gold']['path']}/{schema['gold']['corpus']['label_store']['path']}"

# --- Same as model_training.py ---
def normalize_labels_training(raw):
    if raw is None:
        return []
    parts = re.split(r"[,|;]", str(raw))
    seen, out = set(), []
    for part in parts:
        label = part.strip().lower()
        if label and label not in seen:
            seen.add(label)
            out.append(label)
    return out

# --- Same as label_store.py (semicolon only) ---
def normalize_labels_semicolon(raw):
    if not raw:
        return []
    return sorted({p.strip().lower() for p in str(raw).split(";") if p.strip()})

train_udf = F.udf(normalize_labels_training, ArrayType(StringType()))
semi_udf = F.udf(normalize_labels_semicolon, ArrayType(StringType()))

labels = spark.read.format("delta").load(LABELS_PATH)
train = labels.filter(F.col("category") == "train")

n_docs = train.select("document_id").dropDuplicates().count()

train_norm = (
    train
    .withColumn("target_labels", train_udf(F.col("labels")))
    .filter(F.size("target_labels") > 0)
)

N_train = (
    train_norm
    .select(F.explode("target_labels").alias("lbl"))
    .distinct()
    .count()
)

N_all_splits_training_style = (
    labels
    .withColumn("target_labels", train_udf(F.col("labels")))
    .filter(F.size("target_labels") > 0)
    .select(F.explode("target_labels").alias("lbl"))
    .distinct()
    .count()
)

N_semicolon_train = (
    train
    .withColumn("lbls", semi_udf(F.col("labels")))
    .select(F.explode("lbls").alias("lbl"))
    .distinct()
    .count()
)

print(f"Label store path: {LABELS_PATH}")
print(f"Train documents: {n_docs:,}")
print(f"N on TRAIN (training script — this is what --train-only fits): {N_train:,}")
print(f"N on ALL splits (training-style): {N_all_splits_training_style:,}")
print(f"N on TRAIN (semicolon-only, label_store style): {N_semicolon_train:,}")

if N_train > N_semicolon_train * 2:
    print("\n⚠️  Training N is much larger than semicolon-only — commas in label names may be split incorrectly.")

print("\nTop 30 labels on train (training-style):")
(
    train_norm
    .select(F.explode("target_labels").alias("lbl"))
    .groupBy("lbl")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False)
)

Label store path: s3a://cs611-project/gold/label_store
Train documents: 18,640
N on TRAIN (training script — this is what --train-only fits): 15
N on ALL splits (training-style): 15
N on TRAIN (semicolon-only, label_store style): 12

Top 30 labels on train (training-style):


+-----------------------------------+-----+
|lbl                                |count|
+-----------------------------------+-----+
|trade & customs                    |13254|
|agriculture & food safety          |13110|
|international & eu law             |13079|
|finance                            |4877 |
|tax & banking                      |4877 |
|industry                           |4574 |
|technology & ip                    |4574 |
|economic & competition policy      |3813 |
|constitutional & administrative law|2897 |
|employment & social                |2751 |
|labour                             |2751 |
|environment & energy               |1465 |
|transport & infrastructure         |1427 |
|civil & human rights law           |550  |
|criminal law & security            |353  |
+-----------------------------------+-----+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35476)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.p

In [12]:
from datetime import datetime, timezone
from typing import Any
import pickle
import tempfile

TRAINING_DIR = PROJECT_ROOT / "include" / "training"
if str(TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR))

from model_training import evaluate_multilabel

MP = schema["gold"]["model_predictions"]
PRED_BASE = f"{schema['gold']['path']}/{MP['base']}"
PREFIX = MP["file_prefix"]
SPLIT_COL = "category"
TARGET_LABELS_COL = "target_labels"
PREDICTED_LABELS_COL = "predicted_labels"

# Pin a batch date, or leave None to use the latest prediction_* on R2.
EVAL_DATE = "20260612"  # e.g. "20260612"; None = latest

def load_bytes_from_path(path: str, spark) -> bytes:
    """Read raw bytes from s3a:// via a local temp file (reliable Py4J/Hadoop I/O)."""
    jvm = spark.sparkContext._jvm
    fd, local_path = tempfile.mkstemp()
    os.close(fd)
    try:
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = hadoop_path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
        if not fs.exists(hadoop_path):
            raise FileNotFoundError(path)
        fs.copyToLocalFile(False, hadoop_path, jvm.org.apache.hadoop.fs.Path(local_path))
        with open(local_path, "rb") as handle:
            return handle.read()
    finally:
        if os.path.exists(local_path):
            os.unlink(local_path)

def load_pickle(path: str, spark) -> Any:
    return pickle.loads(load_bytes_from_path(path, spark))

def _list_prediction_child_names(spark, base: str) -> list[str]:
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
    fs = FileSystem.get(URI(base), sc._jsc.hadoopConfiguration())
    p = Path(base)
    if not fs.exists(p):
        return []
    return [st.getPath().getName() for st in fs.listStatus(p)]


def _resolve_eval_date(spark, eval_date: str | None) -> str:
    if eval_date:
        return eval_date
    candidates = sorted(
        name[len(PREFIX) :]
        for name in _list_prediction_child_names(spark, PRED_BASE)
        if name.startswith(PREFIX) and name[len(PREFIX) :].isdigit() and len(name[len(PREFIX) :]) == 8
    )
    if not candidates:
        raise FileNotFoundError(
            f"No prediction batches under {PRED_BASE}. "
            "Run training with --predict-stage predict, then metrics."
        )
    return candidates[-1]

def _label_universe_from_predictions(predictions) -> list[str]:
    labels = (
        predictions.select(F.explode(F.col(TARGET_LABELS_COL)).alias("lbl"))
        .union(predictions.select(F.explode(F.col(PREDICTED_LABELS_COL)).alias("lbl")))
        .select("lbl")
        .distinct()
        .orderBy("lbl")
    )
    return [row.lbl for row in labels.collect()]


def load_prediction_metrics(spark, eval_date: str | None = None) -> tuple[dict, str]:
    """Load metrics from manifest if present; otherwise compute from prediction Delta."""
    batch_date = _resolve_eval_date(spark, eval_date)
    for ext, loader in (
        (".pkl", lambda path: load_pickle(path, spark)),
        (".json", lambda path: json.loads(load_bytes_from_path(path, spark).decode("utf-8"))),
    ):
        path = f"{PRED_BASE}/{PREFIX}{batch_date}{ext}"
        try:
            manifest = loader(path)
            return manifest["metrics"], path
        except FileNotFoundError:
            continue

    delta_path = f"{PRED_BASE}/{PREFIX}{batch_date}"
    delta_names = _list_prediction_child_names(spark, PRED_BASE)
    if f"{PREFIX}{batch_date}" not in delta_names:
        raise FileNotFoundError(f"No prediction outputs for batch {batch_date} under {PRED_BASE}")

    print(f"No manifest found; computing metrics from Delta: {delta_path}")
    predictions = spark.read.format("delta").load(delta_path)
    label_list = _label_universe_from_predictions(predictions)
    print(f"Label universe from predictions: {len(label_list)} labels")
    metrics = evaluate_multilabel(predictions, label_list)
    return metrics, delta_path


metrics, source_path = load_prediction_metrics(spark, EVAL_DATE)
print(f"Metrics source: {source_path}")

for split in ("holdout_val", "holdout_test", "holdout_oot", "holdout_overall"):
    m = metrics[split]
    print(f"\n=== {split} ===")
    print(f"  documents:      {m['documents']}")
    print(f"  micro_f1:        {m['micro_f1']:.4f}")
    print(f"  macro_f1:        {m['macro_f1']:.4f}")
    print(f"  exact_match:     {m['exact_match_ratio']:.4f}")
    print(f"  hamming_loss:    {m['hamming_loss']:.4f}")
    print(f"  micro_p / micro_r: {m['micro_precision']:.4f} / {m['micro_recall']:.4f}")

No manifest found; computing metrics from Delta: s3a://cs611-project/gold/model_predictions/prediction_20260612


Label universe from predictions: 15 labels


[Stage 356:>                                                        (0 + 1) / 1]

Metrics source: s3a://cs611-project/gold/model_predictions/prediction_20260612

=== holdout_val ===
  documents:      5338
  micro_f1:        0.7825
  macro_f1:        0.5662
  exact_match:     0.4262
  hamming_loss:    0.1064
  micro_p / micro_r: 0.8580 / 0.7193

=== holdout_test ===
  documents:      2668
  micro_f1:        0.7850
  macro_f1:        0.5664
  exact_match:     0.4250
  hamming_loss:    0.1054
  micro_p / micro_r: 0.8602 / 0.7218

=== holdout_oot ===
  documents:      1687
  micro_f1:        0.7436
  macro_f1:        0.5098
  exact_match:     0.3871
  hamming_loss:    0.1252
  micro_p / micro_r: 0.8131 / 0.6850

=== holdout_overall ===
  documents:      9693
  micro_f1:        0.7764
  macro_f1:        0.5559
  exact_match:     0.4191
  hamming_loss:    0.1094
  micro_p / micro_r: 0.8508 / 0.7140


## Bronze tables preview

Read Delta tables from R2 and print schema, row counts, and sample rows.

In [3]:
def preview_bronze_table(name: str, path: str, sample_n: int = 5) -> None:
    print("=" * 70)
    print(f"Bronze table: {name}")
    print(f"Path: {path}")
    print("=" * 70)

    df = spark.read.format("delta").load(path)

    print("\nSchema:")
    df.printSchema()

    row_count = df.count()
    print(f"Row count: {row_count:,}")

    print("\nColumns:")
    for col in df.columns:
        print(f"  - {col}")

    if "snapshot_date" in df.columns:
        print("\nSnapshot dates:")
        df.groupBy("snapshot_date").count().orderBy("snapshot_date").show(20, truncate=False)

    if row_count > 0:
        print(f"\nSample rows (first {sample_n}):")
        df.show(sample_n, truncate=80, vertical=False)
    else:
        print("\nTable is empty.")

    print()


preview_bronze_table("legal_docs_raw", LEGAL_BRONZE)
preview_bronze_table("wiki_docs_raw", WIKI_BRONZE)

Bronze table: legal_docs_raw
Path: s3a://cs611-project/bronze/legal_docs_raw



Schema:
root
 |-- CELEX: string (nullable = true)
 |-- Act_name: string (nullable = true)
 |-- Act_type: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- EUROVOC: string (nullable = true)
 |-- Treaty: string (nullable = true)
 |-- Legal_basis_celex: string (nullable = true)
 |-- Authors: string (nullable = true)
 |-- Procedure_number: string (nullable = true)
 |-- Date_document: string (nullable = true)
 |-- Date_publication: string (nullable = true)
 |-- First_entry_into_force: string (nullable = true)
 |-- Temporal_status: string (nullable = true)
 |-- Act_cites: string (nullable = true)
 |-- Cites_links: string (nullable = true)
 |-- Act_ammends: string (nullable = true)
 |-- Ammends_links: string (nullable = true)
 |-- Eurlex_link: string (nullable = true)
 |-- ELI_link: string (nullable = true)
 |-- Proposal_link: string (nullable = true)
 |-- Oeil_link: string (nullable = true)
 |-- Additional_info: string (nullable = true)
 |-- act_raw_text: string (nullable 

Row count: 28,333

Columns:
  - CELEX
  - Act_name
  - Act_type
  - Status
  - EUROVOC
  - Treaty
  - Legal_basis_celex
  - Authors
  - Procedure_number
  - Date_document
  - Date_publication
  - First_entry_into_force
  - Temporal_status
  - Act_cites
  - Cites_links
  - Act_ammends
  - Ammends_links
  - Eurlex_link
  - ELI_link
  - Proposal_link
  - Oeil_link
  - Additional_info
  - act_raw_text
  - labels
  - snapshot_date

Snapshot dates:


+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|1987-01-01   |744  |
|1988-01-01   |787  |
|1989-01-01   |779  |
|1990-01-01   |851  |
|1991-01-01   |911  |
|1992-01-01   |1016 |
|1993-01-01   |1467 |
|1994-01-01   |1725 |
|1995-01-01   |2432 |
|1996-01-01   |2170 |
|1997-01-01   |2235 |
|1998-01-01   |2230 |
|1999-01-01   |2241 |
|2000-01-01   |1243 |
|2001-01-01   |1915 |
|2002-01-01   |2013 |
|2003-01-01   |1887 |
|2004-01-01   |1687 |
+-------------+-----+


Sample rows (first 5):


+----------+--------------------------------------------------------------------------------+----------+------------+--------------------------------------------------------------------------------+--------------+-----------------+-------------------+----------------+-------------+----------------+----------------------+---------------+---------------------+--------------------------------------------------------------------------------+-----------+-------------------------------------------------------+--------------------------------------------------------------------+------------------------------------------+---------------------------------------------------------------------+---------+---------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+-------------+
|     CELEX|                                                                        Act_name|  Act_type|  

+--------+-----------------------------------------------------------------------+--------------------------------+--------------------------------------------------------------------------------+
|      id|                                                                    url|                           title|                                                                            text|
+--------+-----------------------------------------------------------------------+--------------------------------+--------------------------------------------------------------------------------+
|64597748|                     https://en.wikipedia.org/wiki/Bridget%20R.%20Cooks|                Bridget R. Cooks|Bridget R. Cooks is an American scholar, writer, curator, and academic. She i...|
|64557286| https://en.wikipedia.org/wiki/Tennessee%27s%2011th%20Senate%20district|Tennessee's 11th Senate district|Tennessee's 11th Senate district is one of 33 districts in the Tennessee Sena...|
|64556581|     

## Legal docs quality checks

Checks on `legal_docs_raw`:
- missing key fields (`CELEX`, `act_raw_text`, `labels`)
- duplicate `CELEX` IDs
- label distribution
- document length distribution
- gold truncation impact at `MAX_TEXT_CHARS = 500_000` (used by `ngram_processing.py`)
- top raw tokens (before stopword removal / lemmatization)

In [4]:
import re

from pyspark.sql.types import ArrayType, StringType

ID_COL = "CELEX"
TEXT_COL = "act_raw_text"
LABEL_COL = "labels"

legal_df = spark.read.format("delta").load(LEGAL_BRONZE)
total_rows = legal_df.count()
print(f"Total bronze legal rows: {total_rows:,}\n")

# --- 1. Missing documents / missing fields ---
key_df = legal_df.select(
    ID_COL,
    TEXT_COL,
    LABEL_COL,
    (F.col(ID_COL).isNull() | (F.length(F.trim(F.col(ID_COL))) == 0)).alias("missing_id"),
    (F.col(TEXT_COL).isNull() | (F.length(F.trim(F.col(TEXT_COL))) == 0)).alias("missing_text"),
    (F.col(LABEL_COL).isNull() | (F.length(F.trim(F.col(LABEL_COL))) == 0)).alias("missing_label"),
)

missing_id = key_df.filter("missing_id").count()
missing_text = key_df.filter("missing_text").count()
missing_label = key_df.filter("missing_label").count()
missing_any = key_df.filter("missing_id OR missing_text OR missing_label").count()

print("=" * 70)
print("1) Missing field counts")
print("=" * 70)
print(f"  Missing {ID_COL}:      {missing_id:,}")
print(f"  Missing {TEXT_COL}:  {missing_text:,}")
print(f"  Missing {LABEL_COL}:       {missing_label:,}")
print(f"  Missing any key field: {missing_any:,} ({100 * missing_any / total_rows:.2f}%)")

if missing_any > 0:
    print("\nSample rows with missing key fields:")
    key_df.filter("missing_id OR missing_text OR missing_label").show(10, truncate=80)

Total bronze legal rows: 28,333



1) Missing field counts
  Missing CELEX:      0
  Missing act_raw_text:  0
  Missing labels:       0
  Missing any key field: 0 (0.00%)


In [5]:
# --- 2. Duplicate documents ---
dup_df = (
    legal_df.filter(F.col(ID_COL).isNotNull() & (F.length(F.trim(F.col(ID_COL))) > 0))
    .groupBy(ID_COL)
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

dup_celex_count = dup_df.count()
dup_extra_rows = dup_df.agg(F.sum(F.col("count") - 1).alias("extra")).first()["extra"] or 0

print("\n" + "=" * 70)
print("2) Duplicate documents (by CELEX)")
print("=" * 70)
print(f"  CELEX values with duplicates: {dup_celex_count:,}")
print(f"  Extra duplicate rows:         {dup_extra_rows:,}")

if dup_celex_count > 0:
    print("\nTop duplicate CELEX IDs:")
    dup_df.show(15, truncate=False)


2) Duplicate documents (by CELEX)
  CELEX values with duplicates: 0
  Extra duplicate rows:         0


In [6]:
# --- 3. Label distribution ---
print("\n" + "=" * 70)
print("3) Label distribution")
print("=" * 70)

label_dist = (
    legal_df.withColumn(
        LABEL_COL,
        F.when(F.col(LABEL_COL).isNull() | (F.length(F.trim(F.col(LABEL_COL))) == 0), F.lit("(missing)"))
        .otherwise(F.trim(F.col(LABEL_COL))),
    )
    .groupBy(LABEL_COL)
    .count()
    .withColumn("pct", F.round(100 * F.col("count") / total_rows, 2))
    .orderBy(F.desc("count"))
)

label_dist.show(25, truncate=False)


3) Label distribution


+------------------------------------------------------------------------------------------------------------------+-----+-----+
|labels                                                                                                            |count|pct  |
+------------------------------------------------------------------------------------------------------------------+-----+-----+
|Agriculture & Food Safety; Trade & Customs; International & EU Law                                                |6080 |21.46|
|Finance, Tax & Banking; Agriculture & Food Safety; Trade & Customs                                                |2261 |7.98 |
|Industry, Technology & IP; Trade & Customs; International & EU Law                                                |1978 |6.98 |
|Finance, Tax & Banking; Agriculture & Food Safety; Trade & Customs; International & EU Law                        |1379 |4.87 |
|Labour, Employment & Social; Agriculture & Food Safety; Trade & Customs; International & EU Law 

In [7]:
# --- 4. Document length distribution ---
length_df = legal_df.withColumn(
    "char_len",
    F.length(F.trim(F.coalesce(F.col(TEXT_COL), F.lit("")))),
).withColumn(
    "word_len",
    F.when(F.col("char_len") == 0, F.lit(0)).otherwise(F.size(F.split(F.trim(F.col(TEXT_COL)), r"\s+"))),
)

length_stats = length_df.agg(
    F.min("char_len").alias("min_chars"),
    F.max("char_len").alias("max_chars"),
    F.avg("char_len").alias("avg_chars"),
    F.expr("percentile_approx(char_len, 0.5)").alias("median_chars"),
    F.expr("percentile_approx(char_len, 0.95)").alias("p95_chars"),
    F.min("word_len").alias("min_words"),
    F.max("word_len").alias("max_words"),
    F.avg("word_len").alias("avg_words"),
    F.expr("percentile_approx(word_len, 0.5)").alias("median_words"),
).collect()[0]

print("\n" + "=" * 70)
print("4) Document length distribution")
print("=" * 70)
print(
    f"  Characters — min: {length_stats['min_chars']:,}, "
    f"median: {int(length_stats['median_chars']):,}, "
    f"avg: {length_stats['avg_chars']:,.0f}, "
    f"p95: {int(length_stats['p95_chars']):,}, "
    f"max: {length_stats['max_chars']:,}"
)
print(
    f"  Words      — min: {length_stats['min_words']:,}, "
    f"median: {int(length_stats['median_words']):,}, "
    f"avg: {length_stats['avg_words']:,.0f}, "
    f"max: {length_stats['max_words']:,}"
)

length_buckets = (
    length_df.withColumn(
        "char_bucket",
        F.when(F.col("char_len") == 0, "0 (empty)")
        .when(F.col("char_len") < 500, "1-499")
        .when(F.col("char_len") < 1000, "500-999")
        .when(F.col("char_len") < 2500, "1000-2499")
        .when(F.col("char_len") < 5000, "2500-4999")
        .when(F.col("char_len") < 10000, "5000-9999")
        .otherwise("10000+"),
    )
    .groupBy("char_bucket")
    .count()
    .withColumn("pct", F.round(100 * F.col("count") / total_rows, 2))
    .orderBy("char_bucket")
)

print("\nCharacter length buckets:")
length_buckets.show(truncate=False)


4) Document length distribution
  Characters — min: 871, median: 4,076, avg: 10,928, p95: 40,108, max: 2,559,977
  Words      — min: 133, median: 674, avg: 1,791, max: 698,896

Character length buckets:


+-----------+-----+-----+
|char_bucket|count|pct  |
+-----------+-----+-----+
|1000-2499  |4943 |17.45|
|10000+     |6220 |21.95|
|2500-4999  |11727|41.39|
|500-999    |3    |0.01 |
|5000-9999  |5440 |19.2 |
+-----------+-----+-----+



### 4b. Gold truncation cap (`MAX_TEXT_CHARS = 500_000`)

`ngram_processing.py` truncates `act_raw_text` to 500k characters **before** the NLTK UDF runs. This bounds Python worker memory and runtime on EurLex outliers (multi‑million‑char docs) while staying well above typical document sizes (see p95 in section 4).

In [8]:
# --- 4b. Truncation impact at MAX_TEXT_CHARS (gold ngram job) ---
MAX_TEXT_CHARS = 500_000

trunc_df = length_df.withColumn(
    "chars_lost",
    F.greatest(F.col("char_len") - F.lit(MAX_TEXT_CHARS), F.lit(0)),
).withColumn(
    "would_truncate",
    F.col("char_len") > MAX_TEXT_CHARS,
)

trunc_stats = trunc_df.agg(
    F.sum(F.when(F.col("would_truncate"), 1).otherwise(0)).alias("n_truncated"),
    F.count("*").alias("n_total"),
    F.max("char_len").alias("max_chars"),
    F.expr("percentile_approx(char_len, 0.99)").alias("p99_chars"),
    F.sum("chars_lost").alias("total_chars_lost"),
).collect()[0]

n_trunc = int(trunc_stats["n_truncated"])
n_total = int(trunc_stats["n_total"])
pct_trunc = 100.0 * n_trunc / n_total if n_total else 0.0

print("\n" + "=" * 70)
print("4b) Gold truncation cap (MAX_TEXT_CHARS = 500,000)")
print("=" * 70)
print(f"  Documents that would be truncated: {n_trunc:,} / {n_total:,} ({pct_trunc:.2f}%)")
print(f"  p99 char length: {int(trunc_stats['p99_chars']):,}")
print(f"  max char length: {int(trunc_stats['max_chars']):,}")
print(f"  total chars dropped if capped: {int(trunc_stats['total_chars_lost']):,}")

if n_trunc:
    print("\n  Longest documents affected:")
    (
        trunc_df.filter("would_truncate")
        .select(ID_COL, "char_len", "chars_lost")
        .orderBy(F.desc("char_len"))
        .show(10, truncate=False)
    )
else:
    print("\n  No documents exceed the cap on this snapshot.")


4b) Gold truncation cap (MAX_TEXT_CHARS = 500,000)
  Documents that would be truncated: 10 / 28,333 (0.04%)
  p99 char length: 108,188
  max char length: 2,559,977
  total chars dropped if capped: 4,833,540

  Longest documents affected:


+----------+--------+----------+
|CELEX     |char_len|chars_lost|
+----------+--------+----------+
|31998R2261|2559977 |2059977   |
|31994L0055|1361017 |861017    |
|31994D0216|1071130 |571130    |
|31998L0098|927473  |427473    |
|31996R2223|774359  |274359    |
|31997L0024|669387  |169387    |
|32001L0059|665824  |165824    |
|32004D0304|611717  |111717    |
|32004L0073|604683  |104683    |
|31994D0815|587973  |87973     |
+----------+--------+----------+



In [9]:
import re

from pyspark.sql.types import ArrayType, StringType

# --- 5. Top raw tokens ---
# Inline tokenization for Spark UDF (workers cannot import the local `utils` package).
def bronze_raw_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", str(text).lower())

raw_tokenize_udf = F.udf(bronze_raw_tokenize, ArrayType(StringType()))

print("\n" + "=" * 70)
print("5) Top raw tokens (lowercase alphanumeric, no stopword removal yet)")
print("=" * 70)

# Use a sample to avoid Spark OOM on all 58k full-text documents.
TOKEN_SAMPLE_FRACTION = 0.1
text_df = legal_df.filter(
    F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0)
)
sample_df = text_df.sample(withReplacement=False, fraction=TOKEN_SAMPLE_FRACTION, seed=42)
sample_n = sample_df.count()
print(f"Tokenizing {sample_n:,} sampled docs ({TOKEN_SAMPLE_FRACTION:.0%} of non-empty text)")

top_tokens = (
    sample_df.select(F.explode(raw_tokenize_udf(F.col(TEXT_COL))).alias("token"))
    .groupBy("token")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_tokens.show(30, truncate=False)


5) Top raw tokens (lowercase alphanumeric, no stopword removal yet)


Tokenizing 2,761 sampled docs (10% of non-empty text)


+----------+------+
|token     |count |
+----------+------+
|the       |398271|
|of        |229711|
|in        |124305|
|to        |123818|
|and       |105310|
|for       |65272 |
|a         |55516 |
|be        |47799 |
|1         |47599 |
|by        |43097 |
|no        |39687 |
|on        |38792 |
|2         |38671 |
|article   |35147 |
|regulation|33909 |
|is        |31967 |
|that      |31554 |
|as        |30696 |
|or        |30166 |
|3         |29913 |
|shall     |28652 |
|with      |28086 |
|commission|27419 |
|this      |26735 |
|community |21371 |
|4         |21262 |
|from      |21118 |
|not       |20714 |
|which     |20044 |
|l         |19445 |
+----------+------+



## 6. Top tokens after preprocessing

Same sample as section 5. Two code cells mirror production `include/gold/ngram_processing.py`:

- **6a** — `preprocess_tokens_base`: stopword removal + lemmatization
- **6b** — `filter_token_noise`: drop 1-char letters and non-year digit tokens

In [10]:
import re

import nltk
from pyspark.sql.types import ArrayType, StringType

# Download NLTK data on the driver (local Spark workers share the same filesystem in Docker).
for resource in ("stopwords", "wordnet", "omw-1.4"):
    try:
        if resource == "stopwords":
            nltk.data.find("corpora/stopwords")
        elif resource == "wordnet":
            nltk.data.find("corpora/wordnet")
        else:
            nltk.data.find("corpora/omw-1.4")
    except LookupError:
        nltk.download(resource, quiet=True)


def bronze_preprocess_tokens_base(text: str) -> list[str]:
    """Inline copy of ngram_processing.preprocess_tokens_base for Spark workers."""
    if not getattr(bronze_preprocess_tokens_base, "_ready", False):
        from nltk.corpus import stopwords
        from nltk.stem import WordNetLemmatizer

        bronze_preprocess_tokens_base._stops = set(stopwords.words("english"))
        bronze_preprocess_tokens_base._lem = WordNetLemmatizer()
        bronze_preprocess_tokens_base._ready = True

    stops = bronze_preprocess_tokens_base._stops
    lemmatizer = bronze_preprocess_tokens_base._lem
    out: list[str] = []

    for token in re.findall(r"[a-z0-9]+", str(text).lower()):
        if token in stops:
            continue
        out.append(lemmatizer.lemmatize(token))

    return out


preprocess_tokens_base_udf = F.udf(bronze_preprocess_tokens_base, ArrayType(StringType()))

print("\n" + "=" * 70)
print("6a) Top tokens after stopword removal + lemmatization")
print("=" * 70)

if "sample_df" not in globals():
    TOKEN_SAMPLE_FRACTION = 0.1
    text_df = legal_df.filter(
        F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0)
    )
    sample_df = text_df.sample(withReplacement=False, fraction=TOKEN_SAMPLE_FRACTION, seed=42)

print(f"Tokenizing {sample_df.count():,} sampled docs")

top_base_tokens = (
    sample_df.select(F.explode(preprocess_tokens_base_udf(F.col(TEXT_COL))).alias("token"))
    .groupBy("token")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_base_tokens.show(30, truncate=False)


6a) Top tokens after stopword removal + lemmatization


Tokenizing 2,761 sampled docs


+----------+-----+
|token     |count|
+----------+-----+
|1         |47602|
|2         |38676|
|article   |37361|
|regulation|35010|
|3         |29920|
|shall     |28652|
|community |28420|
|commission|27541|
|4         |21263|
|l         |19455|
|ec        |18511|
|state     |17972|
|5         |17725|
|eec       |17256|
|p         |17085|
|product   |16995|
|member    |16961|
|european  |15568|
|10        |14860|
|whereas   |14249|
|oj        |13926|
|6         |13047|
|market    |12208|
|decision  |12202|
|7         |11469|
|aid       |10843|
|annex     |10805|
|may       |10298|
|measure   |10203|
|regard    |10107|
+----------+-----+



In [11]:
def bronze_filter_token_noise(tokens: list[str]) -> list[str]:
    """Inline copy of ngram_processing.filter_token_noise for Spark workers."""
    out: list[str] = []

    for token in tokens:
        if len(token) == 1 and token.isalpha():
            continue
        if token.isdigit():
            if len(token) != 4:
                continue
            out.append(token)
            continue
        out.append(token)

    return out


filter_token_noise_udf = F.udf(bronze_filter_token_noise, ArrayType(StringType()))

print("\n" + "=" * 70)
print("6b) Top tokens after noise filter (1-char letters + non-year digits removed)")
print("=" * 70)

if "sample_df" not in globals():
    TOKEN_SAMPLE_FRACTION = 0.1
    text_df = legal_df.filter(
        F.col(TEXT_COL).isNotNull() & (F.length(F.trim(F.col(TEXT_COL))) > 0)
    )
    sample_df = text_df.sample(withReplacement=False, fraction=TOKEN_SAMPLE_FRACTION, seed=42)

print(f"Tokenizing {sample_df.count():,} sampled docs")

top_filtered_tokens = (
    sample_df.select(
        F.explode(
            filter_token_noise_udf(preprocess_tokens_base_udf(F.col(TEXT_COL)))
        ).alias("token")
    )
    .groupBy("token")
    .count()
    .orderBy(F.desc("count"))
    .limit(30)
)

top_filtered_tokens.show(30, truncate=False)


6b) Top tokens after noise filter (1-char letters + non-year digits removed)


Tokenizing 2,761 sampled docs


+----------+-----+
|token     |count|
+----------+-----+
|article   |37361|
|regulation|35010|
|shall     |28652|
|community |28420|
|commission|27541|
|ec        |18511|
|state     |17972|
|eec       |17256|
|product   |16995|
|member    |16961|
|european  |15568|
|whereas   |14249|
|oj        |13926|
|market    |12208|
|decision  |12202|
|aid       |10843|
|annex     |10805|
|may       |10298|
|measure   |10203|
|regard    |10107|
|price     |9537 |
|council   |9325 |
|import    |8720 |
|directive |8266 |
|official  |8101 |
|must      |8098 |
|accordance|7407 |
|within    |6776 |
|particular|6743 |
|period    |6729 |
+----------+-----+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 43222)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/accumulators.p